# 🤖 Requirements Research Agent
### CAS Application Development with AI (ADAI) · Modul 2
**BFH Biel · Ilja Rasin**

---

Dieser Agent analysiert euer Projektumfeld automatisch:

1. **Marktrecherche** — sucht ähnliche Produkte im Web
2. **Feature-Analyse** — was haben alle? Was haben wenige? Was fehlt überall?
3. **MoSCoW-Empfehlung** — basierend auf echten Marktdaten
4. **Gaps finden** — was machen Konkurrenten, das ihr vergessen habt?

> **Wichtig:** Dies ist ein *Lernbeispiel*. Der Agent macht Vorschläge — euer Team entscheidet.

## Schritt 1 — Installation

Wir brauchen nur die Anthropic Library. Claude übernimmt die Websuche via Tool Use.

In [16]:
# Anthropic SDK installieren
!pip install anthropic -q
print('✅ Installation abgeschlossen')


[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
✅ Installation abgeschlossen


## Schritt 2 — API Key

Ihr braucht einen Anthropic API Key.
→ https://console.anthropic.com → API Keys → Create Key

> **Sicherheit:** Niemals den API Key in Code einfügen der geteilt wird!
> In Colab: Secrets (🔑 links) → Name: `ANTHROPIC_API_KEY`

In [17]:
import os
import json
import anthropic
import httpx
# ── API Key ────────────────────────────────────────────────
try:
    from google.colab import userdata
    api_key = userdata.get('ANTHROPIC_API_KEY')
    print('✅ API Key aus Colab Secrets geladen')
except Exception:
    api_key = os.getenv('ANTHROPIC_API_KEY') or input('Anthropic API Key eingeben: ')
# ── Proxy + CA-Bundle ──────────────────────────────────────
# Corporate proxy (EJPD / Infomaniak Intranet)
PROXY    = os.getenv('HTTPS_PROXY', 'http://proxy.infet.ejpd.admin.ch:8080')
CA_BUNDLE = os.getenv('SSL_CERT_FILE', '/etc/ssl/certs/ca-certificates.crt')
http_client = httpx.Client(
    proxy=PROXY,
    verify=CA_BUNDLE,
    timeout=httpx.Timeout(90.0),   # 90s – mehr Puffer fuer Corporate-Proxy-Timeout
)
client = anthropic.Anthropic(
    api_key=api_key,
    http_client=http_client,
)
print(f'✅ Client initialisiert')
print(f'   🔒 CA-Bundle : {CA_BUNDLE}')
print(f'   🌐 Proxy     : {PROXY}')


✅ Client initialisiert
   🔒 CA-Bundle : /etc/ssl/certs/ca-certificates.crt
   🌐 Proxy     : http://proxy.infet.ejpd.admin.ch:8080


## Schritt 3 — Euer Projekt beschreiben

Hier tragt ihr euer Projekt ein. Der Agent nutzt diese Informationen
als Grundlage für die Marktrecherche.

**Füllt alle Felder aus — je mehr Details, desto besser die Analyse.**

In [19]:
# ============================================================
# HIER EUER PROJEKT EINTRAGEN
# ============================================================
from pathlib import Path
def read_docs_markdown() -> str:
    """Liest alle Markdown-Dateien aus docs/ als Projektkontext ein."""
    candidate_dirs = [Path.cwd() / "docs", Path.cwd().parent / "docs"]
    docs_dir = next((d for d in candidate_dirs if d.exists()), None)
    if docs_dir is None:
        raise FileNotFoundError("Konnte kein docs/-Verzeichnis finden.")
    markdown_files = sorted(docs_dir.glob("*.md"))
    if not markdown_files:
        raise FileNotFoundError(f"Keine Markdown-Dateien in {docs_dir} gefunden.")
    return "\n\n".join(
        f"### {md_file.name}\n{md_file.read_text(encoding='utf-8')}" for md_file in markdown_files
    )
projekt_beschreibung = read_docs_markdown()
PROJEKT = {
    # Projektname
    "name": "JobMatch",
    # Kontext aus docs/*.md statt manuell als String
    "beschreibung": projekt_beschreibung,
    # Zielgruppe
    "zielgruppe": "Recruiter, Hiring Manager, IT Consulting Unternehmen",
    # Eure aktuellen User Stories (kurz)
    "user_stories": """
     1.  As an **account manager**, I want to see a per-axis score breakdown (seniority, stack, domain, languages, position, region) for each ranked employee, so that I can understand why a candidate was ranked highly and explain the choice confidently to the client.

**Acceptance Criteria:**
- Given a ranked list is displayed, when I view any employee entry, then I can see individual scores for seniority, stack, domain, and language axes alongside the overall rank.
- Given an employee has a low score on one axis, when I view their breakdown, then the underperforming axis is visually distinguishable from the others.

2. As an **account manager**, I want to read a human-readable explanation for each employee's match, so that I can quickly verify the reasoning before shortlisting and pass it on to the client.

**Acceptance Criteria:**
- Given a ranked list is displayed, when I view an employee entry, then a short natural-language explanation of the match is shown alongside the scores.
- Given the explanation is shown, when I read it, then it references specific aspects of both the job offer and the employee's CV rather than generic filler text.


3. As a **dev team member**, I want the ingestion pipeline to extract structured fields (roles, skills, education, languages, domains) from raw CV and JD text, so that the matching pipeline operates on normalized, comparable data.

**Acceptance Criteria:**
- Given a raw CV or JD is submitted to the ingestion pipeline, when extraction completes, then all defined structured fields are returned in a consistent schema.
- Given a multilingual document is submitted, when extraction completes, then field values are extracted correctly regardless of the document's source language.
- Given the extraction step calls an LLM, when the call is made, then only pseudonymized text (US-T09) is sent, and the inference endpoint is Swiss-hosted under a documented data processing agreement (real or template) referenced in the ADRs.

    """,
    # Bereich / Branche
    "branche": "HR Tech, Recruiting, IT Consulting",
    # Land / Markt
    "markt": "Schweiz"
}
print(f'✅ Projekt konfiguriert: {PROJEKT["name"]}')
print(f'📁 docs-Kontext geladen aus Markdown-Dateien: {len(projekt_beschreibung)} Zeichen')
print(f'📋 Branche: {PROJEKT["branche"]}')
print(f'🌍 Markt: {PROJEKT["markt"]}')



✅ Projekt konfiguriert: JobMatch
📁 docs-Kontext geladen aus Markdown-Dateien: 37060 Zeichen
📋 Branche: HR Tech, Recruiting, IT Consulting
🌍 Markt: Schweiz


## Schritt 4 — Agent-Architektur

So funktioniert unser Agent:

```
┌─────────────────────────────────────────────┐
│              REQUIREMENTS AGENT              │
│                                              │
│  1. Marktrecherche     → web_search Tool     │
│  2. Konkurrenzanalyse  → web_search Tool     │
│  3. Feature-Matrix     → analyze Tool        │
│  4. MoSCoW-Empfehlung  → Reasoning           │
│  5. Gaps-Report        → Reasoning           │
└─────────────────────────────────────────────┘
```

Der Agent entscheidet selbst **wann** er sucht und **was** er sucht.
Wir geben ihm nur das Ziel — er plant den Weg.

In [20]:
# ============================================================
# TOOL-DEFINITIONEN
# Der Agent kann diese Tools verwenden
# ============================================================

tools = [
    {
        "name": "web_search",
        "description": """
            Sucht im Web nach aktuellen Informationen.
            Verwende dieses Tool um:
            - Ähnliche Produkte / Konkurrenten zu finden
            - Features von bestehenden Lösungen zu recherchieren
            - Markttrends zu identifizieren
            - Branchenstandards zu verstehen
        """,
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Suchanfrage auf Englisch für bessere Ergebnisse"
                },
                "purpose": {
                    "type": "string",
                    "description": "Warum wird gesucht? (z.B. 'Konkurrenten finden')"
                }
            },
            "required": ["query", "purpose"]
        }
    },
    {
        "name": "analyze_features",
        "description": """
            Analysiert gesammelte Features und erstellt eine
            strukturierte Feature-Matrix für MoSCoW-Priorisierung.
        """,
        "input_schema": {
            "type": "object",
            "properties": {
                "gefundene_features": {
                    "type": "string",
                    "description": "Liste der gefundenen Features aus der Recherche"
                },
                "eigene_stories": {
                    "type": "string",
                    "description": "Die User Stories des Projekts"
                }
            },
            "required": ["gefundene_features", "eigene_stories"]
        }
    }
]

print('✅ Tools definiert:')
for tool in tools:
    print(f'  🔧 {tool["name"]}')

✅ Tools definiert:
  🔧 web_search
  🔧 analyze_features


In [21]:
# ============================================================
# TOOL-AUSFHRUNG
# Was passiert wenn der Agent ein Tool aufruft
# ============================================================
import os
import time
import anthropic
import httpx
# Retry-Konfiguration (per Env ueberschreibbar)
MAX_API_RETRIES = int(os.getenv('ANTHROPIC_MAX_RETRIES', '4'))
INITIAL_BACKOFF_SEC = float(os.getenv('ANTHROPIC_RETRY_BACKOFF_SEC', '1.5'))
def create_message_with_retry(**kwargs):
    """Robuster Wrapper fuer client.messages.create mit Retry bei Verbindungsproblemen."""
    backoff = INITIAL_BACKOFF_SEC
    for attempt in range(1, MAX_API_RETRIES + 1):
        try:
            return client.messages.create(**kwargs)
        except anthropic.APIStatusError as exc:
            retryable = exc.status_code in (408, 409, 429) or exc.status_code >= 500
            if not retryable or attempt == MAX_API_RETRIES:
                raise
            print(f'  ⚠️ APIStatusError {exc.status_code}; Retry {attempt}/{MAX_API_RETRIES} in {backoff:.1f}s')
        except (anthropic.APIConnectionError, anthropic.APITimeoutError, httpx.TimeoutException, httpx.TransportError) as exc:
            if attempt == MAX_API_RETRIES:
                raise RuntimeError(
                    'Anthropic API weiterhin nicht erreichbar. '
                    'Bitte Netzwerk/Proxy/CA-Bundle pruefen (ANTHROPIC_CA_BUNDLE oder SSL_CERT_FILE).'
                ) from exc
            print(f'  ⚠️ Verbindungsfehler; Retry {attempt}/{MAX_API_RETRIES} in {backoff:.1f}s')
            # Stale Verbindung verwerfen und neuen httpx-Client erstellen
            client.http_client = httpx.Client(
                proxy=PROXY,
                verify=CA_BUNDLE,
                timeout=httpx.Timeout(90.0),
            )
        time.sleep(backoff)
        backoff *= 2
# Simulierte Websuche (in Produktion: echte Search API)
# Fr den Kurs nutzen wir Claude selbst als 'Wissensquelle'
search_cache = {}
def execute_web_search(query: str, purpose: str) -> str:
    """
    Fhrt eine Websuche durch.
    In diesem Demo: Claude simuliert Suchergebnisse.
    In Produktion: Serper API, Brave Search, etc.
    """
    print(f'  🔍 Suche: "{query}"')
    print(f'     Zweck: {purpose}')
    # Claude als Wissensbasis nutzen
    response = create_message_with_retry(
        model='claude-sonnet-4-6',
        max_tokens=4000,
        messages=[{
            'role': 'user',
            'content': f"""
                Du bist eine Suchmaschine. Beantworte diese Suchanfrage
                mit konkreten, realistischen Informationen:
                Suchanfrage: {query}
                Kontext: {purpose}
                Antworte mit:
                - 3-5 konkrete Beispiele / Produkte / Features
                - Kurze, faktische Beschreibungen
                - Keine Einleitung, direkt zur Sache
            """
        }]
    )
    result = response.content[0].text
    print(f'  ✅ Ergebnis erhalten ({len(result)} Zeichen)')
    return result
def execute_analyze_features(gefundene_features: str, eigene_stories: str) -> str:
    """
    Analysiert Features und erstellt Feature-Matrix.
    """
    print('  📊 Analysiere Feature-Matrix...')
    response = create_message_with_retry(
        model='claude-sonnet-4-6',
        max_tokens=4000,
        messages=[{
            'role': 'user',
            'content': f"""
                Erstelle eine Feature-Matrix basierend auf:
                Gefundene Markt-Features:
                {gefundene_features}
                Eigene User Stories:
                {eigene_stories}
                Kategorisiere Features in:
                - BERALL vorhanden (→ Must have)
                - HUFIG vorhanden (→ Should have)
                - SELTEN vorhanden (→ Could have)
                - KAUM vorhanden (→ Won't have / Innovation)
                - IN EIGENEN STORIES FEHLEND (→ Gap!)
                Antworte strukturiert mit klaren Kategorien.
            """
        }]
    )
    result = response.content[0].text
    print('  ✅ Analyse abgeschlossen')
    return result
def execute_tool(tool_name: str, tool_input: dict) -> str:
    """Router: fhrt das richtige Tool aus."""
    if tool_name == 'web_search':
        return execute_web_search(
            query=tool_input['query'],
            purpose=tool_input['purpose']
        )
    elif tool_name == 'analyze_features':
        return execute_analyze_features(
            gefundene_features=tool_input['gefundene_features'],
            eigene_stories=tool_input['eigene_stories']
        )
    else:
        return f'Unbekanntes Tool: {tool_name}'
print('✅ Tool-Ausfhrung konfiguriert')


✅ Tool-Ausfhrung konfiguriert


## Schritt 5 — Der Agent

Das Herzstück: Der **Agentic Loop**.

```
Agent denkt → ruft Tool auf → bekommt Ergebnis
      ↑                                    ↓
      └──────── denkt weiter ──────────────┘
```

Dies wiederholt sich bis der Agent fertig ist (`stop_reason == 'end_turn'`).

In [22]:
# ============================================================
# AGENTIC LOOP
# Der Kern des Agenten
# ============================================================
def run_requirements_agent(projekt: dict) -> str:
    """
    Fuehrt den Requirements Research Agent aus.
    Der Agent:
    1. Plant seine Recherche-Strategie
    2. Fuehrt mehrere Websuchen durch
    3. Analysiert die Ergebnisse
    4. Erstellt MoSCoW-Empfehlung
    5. Identifiziert Gaps
    """
    # System Prompt: Was soll der Agent tun?
    system_prompt = f"""
    Du bist ein Requirements Research Agent fuer Software-Projekte.
    Deine Aufgabe:
    1. Recherchiere den Markt fuer das gegebene Projekt
    2. Finde 3-5 aehnliche Produkte / Konkurrenten
    3. Analysiere welche Features ueberall, haeufig oder selten sind
    4. Erstelle MoSCoW-Empfehlung basierend auf Marktdaten
    5. Identifiziere Gaps in den User Stories
    Arbeite systematisch:
    - Erst Konkurrenten suchen
    - Dann Features analysieren
    - Dann MoSCoW erstellen
    - Zuletzt Gaps identifizieren
    Schreibe deinen finalen Report auf DEUTSCH.
    Sei konkret - keine vagen Aussagen.
    """

    # Erste Nachricht: Projektbeschreibung
    initial_message = f"""
    Analysiere dieses Projekt:
    **Projektname:** {projekt['name']}
    **Beschreibung:** {projekt['beschreibung']}
    **Zielgruppe:** {projekt['zielgruppe']}
    **Branche:** {projekt['branche']}
    **Markt:** {projekt['markt']}

    **Unsere aktuellen User Stories:**
    {projekt['user_stories']}

    Starte deine Analyse. Fuehre zuerst Websuchen durch,
    dann analysiere und erstelle den Report.
    """

    messages = [{"role": "user", "content": initial_message}]
    report_parts = []

    print('Agent startet...\n')
    print('=' * 60)

    iteration = 0
    max_iterations = int(os.getenv('AGENT_MAX_ITERATIONS', '12'))
    response_max_tokens = int(os.getenv('AGENT_RESPONSE_MAX_TOKENS', '4000'))

    # -- AGENTIC LOOP ---------------------------------------
    while iteration < max_iterations:
        iteration += 1
        print(f'\nIteration {iteration}')

        response = create_message_with_retry(
            model="claude-sonnet-4-6",
            max_tokens=response_max_tokens,
            system=system_prompt,
            tools=tools,
            messages=messages
        )

        text_blocks = [block.text for block in response.content if hasattr(block, 'text') and block.text]
        assistant_content = [block.model_dump() for block in response.content]
        messages.append({"role": "assistant", "content": assistant_content})

        if response.stop_reason == "end_turn":
            print('\nAgent fertig!')
            print('=' * 60)
            if report_parts:
                report_parts.extend(text_blocks)
                return '\n\n'.join(part.strip() for part in report_parts if part and part.strip())
            for block in response.content:
                if hasattr(block, 'text'):
                    return block.text
            return "Agent abgeschlossen - kein Text-Output"

        if response.stop_reason == "max_tokens":
            print('\nAntwort wegen max_tokens abgeschnitten - setze fort...')
            report_parts.extend(text_blocks)
            messages.append({
                "role": "user",
                "content": "Bitte fahre exakt an der letzten Stelle fort. Wiederhole nichts und beende den finalen Report vollstaendig."
            })
            time.sleep(0.5)
            continue

        if response.stop_reason == "tool_use":
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f'\nTool aufgerufen: {block.name}')
                    result = execute_tool(block.name, block.input)
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result
                    })

            if not tool_results:
                print('\nWarnung: stop_reason=tool_use, aber keine Tool-Aufrufe gefunden - Abbruch')
                break

            messages.append({
                "role": "user",
                "content": tool_results
            })
            time.sleep(0.5)
            continue

        print(f'\nWarnung: Unbekannter stop_reason: {response.stop_reason} - Abbruch')
        break

    if report_parts:
        return '\n\n'.join(part.strip() for part in report_parts if part and part.strip())
    return "Maximum Iterationen erreicht"

print('Agent-Funktion bereit')
print('   -> Naechste Zelle: Agent starten!')


Agent-Funktion bereit
   -> Naechste Zelle: Agent starten!


## Schritt 6 — Agent starten! 🚀

Jetzt läuft der Agent. Ihr seht live:
- Welche Tools er aufruft
- Was er sucht
- Wie er iteriert

**Dauer:** ca. 1-2 Minuten

In [23]:
# ============================================================
# AGENT STARTEN
# ============================================================

report = run_requirements_agent(PROJEKT)

print('\n')
print('=' * 60)
print('📄 REQUIREMENTS RESEARCH REPORT')
print('=' * 60)
print(report)

Agent startet...


Iteration 1

Tool aufgerufen: web_search
  🔍 Suche: "AI-powered CV job matching software HR tech 2024 competitors"
     Zweck: Konkurrenten finden
  ✅ Ergebnis erhalten (1204 Zeichen)

Tool aufgerufen: web_search
  🔍 Suche: "semantic CV matching software recruiting SaaS features comparison 2024"
     Zweck: Features von ähnlichen Produkten recherchieren
  ✅ Ergebnis erhalten (1536 Zeichen)

Tool aufgerufen: web_search
  🔍 Suche: "internal talent matching platform IT consulting employee skills HR software"
     Zweck: Spezifische Konkurrenten im IT-Consulting Bereich finden
  ✅ Ergebnis erhalten (1371 Zeichen)

Iteration 2

Tool aufgerufen: web_search
  🔍 Suche: "LLM-as-judge CV ranking explainable AI matching recruiter features 2024"
     Zweck: Features für erklärbare KI im CV-Matching recherchieren
  ✅ Ergebnis erhalten (1847 Zeichen)

Tool aufgerufen: web_search
  🔍 Suche: "HR matching software bias detection fairness scoring candidate ranking features"
     Zweck

## Schritt 7 — Report speichern

In [24]:
# ============================================================
# REPORT ALS MARKDOWN SPEICHERN
# ============================================================

filename = f"requirements_report_{PROJEKT['name'].lower().replace(' ', '_')}.md"

with open(filename, 'w', encoding='utf-8') as f:
    f.write(f"# Requirements Research Report\n")
    f.write(f"## {PROJEKT['name']}\n")
    f.write(f"*Generiert von Requirements Research Agent · ADAI 2026*\n\n")
    f.write(report)

print(f'✅ Report gespeichert: {filename}')

# In Colab: Download
try:
    from google.colab import files
    files.download(filename)
    print('📥 Download gestartet')
except:
    print(f'📁 Datei liegt im aktuellen Verzeichnis: {filename}')

✅ Report gespeichert: requirements_report_jobmatch.md
📁 Datei liegt im aktuellen Verzeichnis: requirements_report_jobmatch.md


## Schritt 8 — Euer eigenes Projekt analysieren

Jetzt seid ihr dran! Ändert die Werte in **Schritt 3** auf euer Projekt
und führt die Zellen 3 → 6 → 7 erneut aus.

---

## Was wir heute gelernt haben

| Konzept | Erklärung |
|---------|----------|
| **Agentic Loop** | Agent denkt → Tool → Ergebnis → weiterdenken |
| **Tool Use** | Agent entscheidet selbst wann und was er sucht |
| **stop_reason** | `tool_use` = weiter, `end_turn` = fertig |
| **Messages History** | Agent braucht Kontext aller vorherigen Schritte |
| **System Prompt** | Definiert Persönlichkeit und Aufgabe des Agenten |

---

## Diskussion

- Was macht der Agent besser als manuelles Prompting?
- Wo ist er schlechter? Wo halluziniert er?
- Wie würdet ihr ihn verbessern?
- Wofür könnt ihr Agenten in **eurem** Projekt einsetzen?

---
*CAS ADAI 2026 · BFH Biel · Modul 2 · Ilja Rasin*